<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Qwen3_Inference_and_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [How Well Does Qwen3 Handle 4-bit and 2-bit Quantization?](https://kaitchup.substack.com/p/how-well-does-qwen3-handle-4-bit)*

This notebook shows:
* How to run Qwen3 models with vLLM, with and without reasoning turned on
* How to quantize Qwen3 to 4-bit and 2-bit in formats optimized for GPUs (GPTQ)
* How to evaluate the models

# Installation

I installed vLLM from source to disable reasoning with chat_template_kwargs. This will be unnecessary from vLLM 0.8.6

In [ ]:
!git clone https://github.com/vllm-project/vllm.git && cd vllm && VLLM_USE_PRECOMPILED=1 pip install --editable .

# Inference with vLLM and 2-bit Qwen3 32B

In [ ]:
%env VLLM_USE_V1=0
from vllm.vllm import LLM, SamplingParams

# Sample prompts.
prompts = [[{"role": "user", "content": "Let a, b be positive real numbers such that ab = a + b + 3. Determine the range of possible values for a + b."}]]

# Create a sampling params object.
sampling_params = SamplingParams(temperature=0.6, top_k=20, top_p=0.95, max_tokens=8192)

# Create an LLM.
llm = LLM(model="kaitchup/Qwen3-32B-autoround-2bit-gptq")
# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.chat(prompts, sampling_params, chat_template_kwargs={"enable_thinking": False})
# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

outputs = llm.chat(prompts, sampling_params, chat_template_kwargs={"enable_thinking": True})
# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

env: VLLM_USE_V1=0
INFO 05-01 00:47:55 [__init__.py:239] Automatically detected platform cuda.
INFO 05-01 00:48:04 [config.py:749] This model supports multiple tasks: {'embed', 'reward', 'generate', 'classify', 'score'}. Defaulting to 'generate'.
WARNING 05-01 00:48:05 [config.py:863] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 05-01 00:48:05 [arg_utils.py:1371] Chunked prefill is enabled by default for models with max_model_len > 32K. Chunked prefill might not work with some features or models. If you encounter any issues, please disable by launching with --enable-chunked-prefill=False.
INFO 05-01 00:48:05 [config.py:2036] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 05-01 00:48:05 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5.dev380+g200bbf92e) with config: model='kaitchup/Qwen3-32B-autoround-2bit-gptq', speculative_config=None, tokenizer='kaitchup/Qwen3-32B-autoround-2bit-gptq', skip_tokeniz

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-01 00:48:12 [loader.py:458] Loading weights took 4.58 seconds
INFO 05-01 00:48:12 [model_runner.py:1140] Model loading took 12.2861 GiB and 5.431664 seconds
INFO 05-01 00:48:14 [worker.py:287] Memory profiling takes 1.20 seconds
INFO 05-01 00:48:14 [worker.py:287] the current vLLM instance can use total_gpu_memory (79.26GiB) x gpu_memory_utilization (0.90) = 71.33GiB
INFO 05-01 00:48:14 [worker.py:287] model weights take 12.29GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 57.55GiB.
INFO 05-01 00:48:14 [executor_base.py:112] # cuda blocks: 14731, # CPU blocks: 1024
INFO 05-01 00:48:14 [executor_base.py:117] Maximum concurrency for 40960 tokens per request: 5.75x
INFO 05-01 00:48:17 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CL

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 05-01 00:48:43 [model_runner.py:1592] Graph capturing finished in 26 secs, took 1.24 GiB
INFO 05-01 00:48:43 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 31.19 seconds
INFO 05-01 00:48:44 [chat_utils.py:397] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: None, Generated text: "We are given the equation:\n\n$$\nab = a + b + 3\n$$\n\nLet us define $ a + b = S $ and $ ab = P $. Then the equation becomes:\n\n$$\nP = S + 3\n$$\n\nWe want to find the range of possible values for $ S = a + b $.\n\n---\n\n### Step 1: Express $ P $ in terms of $ S $\n\nFrom the equation $ ab = a + b + 3 $, we have:\n\n$$\nP = S + 3\n$$\n\nLet’s also recall the identity:\n\n$$\na + b = S \\quad \\text{and} \\quad ab = P\n$$\n\nWe can use the identity:\n\n$$\na^2 + b^2 = S^2 - 2P\n$$\n\nBut we don't need that now. Instead, let's try to find the range of $ S $.\n\n---\n\n### Step 2: Express $ a $ and $ b $ in terms of $ S $\n\nLet’s define $ a = x $ and $ b = y $. Then:\n\n$$\nx + y = S \\quad \\text{and} \\quad xy = P = S + 3\n$$\n\nLet’s solve for $ x $ and $ y $ in terms of $ S $.\n\nWe can write the quadratic equation in terms of $ x $:\n\n$$\nx^2 + x(y) = S \\quad \\text{and} \\quad x(y) = P = S + 3\n$$\n\nLet’s substitute $ y = S - x $ into the equat

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: None, Generated text: "Okay, so I need to figure out the range of possible values for a + b given that ab = a + b + 3, where a and b are positive real numbers. Let me start by trying to understand the problem and how to approach it.\n\nFirst, the equation is ab = a + b + 3. I need to find the range of a + b. Let me denote S = a + b and P = ab. Then the equation becomes P = S + 3. So if I can express S in terms of P, or vice versa, maybe I can find some relationship between them.\n\nBut since S = a + b and P = ab, I know that for two variables a and b, there are relationships between S and P. For example, if I consider a and b as roots of a quadratic equation, then S would be the sum of the roots and P the product. However, in this case, maybe I don't need to go that route. Let me try to manipulate the equation.\n\nGiven that ab = a + b + 3, let me substitute S = a + b into the equation. Then we have ab = S + 3. So P = S + 3. But also, since S = a + b, we can express P in terms 

# Evaluation

The Evaluation Harness must also be installed from source.

In [ ]:
!git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness && cd lm-evaluation-harness && pip install -e .

Then, the vLLM calls must be modified to disable reasoning if you use benchmarks not designed for reasoning.

Modify this file in lm-eval: ./ models/ vllm_causallms.py to add "enable_thinking=False" to apply_chat_template.

In [ ]:
!VLLM_USE_V1=0 lm_eval --model vllm \
    --model_args pretrained="Qwen/Qwen3-32B",dtype="bfloat16",max_model_len=12000 \
    --tasks leaderboard_ifeval \
    --device cuda:0 \
    --batch_size auto \
    --apply_chat_template \
    --output_path results

Loading safetensors checkpoint shards:  18% Completed | 3/17 [00:02<00:11,  1.21it/s]
Loading safetensors checkpoint shards:  24% Completed | 4/17 [00:03<00:10,  1.19it/s]
Loading safetensors checkpoint shards:  29% Completed | 5/17 [00:04<00:10,  1.19it/s]
Loading safetensors checkpoint shards:  35% Completed | 6/17 [00:05<00:09,  1.18it/s]
Loading safetensors checkpoint shards:  41% Completed | 7/17 [00:05<00:08,  1.17it/s]
Loading safetensors checkpoint shards:  47% Completed | 8/17 [00:06<00:07,  1.18it/s]
Loading safetensors checkpoint shards:  53% Completed | 9/17 [00:07<00:06,  1.20it/s]
Loading safetensors checkpoint shards:  59% Completed | 10/17 [00:08<00:06,  1.11it/s]
Loading safetensors checkpoint shards:  65% Completed | 11/17 [00:09<00:05,  1.14it/s]
Loading safetensors checkpoint shards:  71% Completed | 12/17 [00:10<00:04,  1.06it/s]
Loading safetensors checkpoint shards:  76% Completed | 13/17 [00:11<00:03,  1.01it/s]
Loading safetensors checkpoint shards:  82% Comple

In [ ]:
!VLLM_USE_V1=0 lm_eval --model vllm \
    --model_args pretrained="kaitchup/Qwen3-32B-autoround-4bit-gptq",dtype="float16",max_model_len=12000 \
    --tasks leaderboard_ifeval \
    --device cuda:0 \
    --batch_size auto \
    --apply_chat_template \
    --output_path results

INFO 04-30 23:11:55 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 04-30 23:11:55 [__init__.py:239] Automatically detected platform cuda.
2025-04-30:23:11:58 INFO     [__main__:440] Selected Tasks: ['leaderboard_ifeval']
2025-04-30:23:11:58 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-04-30:23:11:58 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'kaitchup/Qwen3-32B-autoround-4bit-gptq', 'dtype': 'float16', 'max_model_len': 12000}
config.json: 100%|█████████████████████████| 1.55k/1.55k [00:00<00:00, 20.2MB/s]
INFO 04-30 23:12:06 [config.py:749] This model supports multiple tasks: {'embed', 'score', 'classify', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 04-30 23:12:07 [gptq_marlin.py:144] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 04-30 23:12:07 [llm_engine.py:

In [ ]:
!VLLM_USE_V1=0 lm_eval --model vllm \
    --model_args pretrained="kaitchup/Qwen3-32B-autoround-2bit-gptq",dtype="float16",max_model_len=12000 \
    --tasks leaderboard_ifeval \
    --device cuda:0 \
    --batch_size auto \
    --apply_chat_template \
    --output_path results

INFO 04-30 23:04:11 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 04-30 23:04:11 [__init__.py:239] Automatically detected platform cuda.
2025-04-30:23:04:15 INFO     [__main__:440] Selected Tasks: ['leaderboard_ifeval']
2025-04-30:23:04:15 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-04-30:23:04:15 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'kaitchup/Qwen3-32B-autoround-2bit-gptq', 'dtype': 'float16', 'max_model_len': 12000}
config.json: 100%|█████████████████████████| 1.55k/1.55k [00:00<00:00, 9.45MB/s]
INFO 04-30 23:04:23 [config.py:749] This model supports multiple tasks: {'embed', 'reward', 'score', 'generate', 'classify'}. Defaulting to 'generate'.
WARNING 04-30 23:04:24 [config.py:863] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 04-30 23:04:24 [l

# Quantization

## 2-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "Qwen/Qwen3-4B"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
from auto_round import AutoRound

autoround = AutoRound(model, tokenizer, nsamples=512, iters=512, low_gpu_mem_usage=False, enable_torch_compile=True, bits=2, seqlen=4096, group_size=32, sym=True)
output_dir = "./Qwen3-4B-autoround-2bit-gptq"
autoround.quantize_and_save(output_dir, format='auto_gptq')



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2025-04-30 23:21:45 INFO autoround.py L240: using torch.bfloat16 for quantization tuning
2025-04-30 23:21:45 INFO autoround.py L570: start to cache block inputs
2025-04-30 23:21:47,475 INFO config.py L54: PyTorch version 2.7.0 available.


README.md:   0%|          | 0.00/373 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

(…)-00000-of-00001-4746b8785c874cc7.parquet:   0%|          | 0.00/33.3M [00:00<?, ?B/s]

...

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/728 [00:00<?, ? examples/s]

2025-04-30 23:22:30 INFO autoround.py L575: caching done
Quantizing model.layers.0:   0%|          | 0/36 [00:03<?, ?it/s]W0430 23:22:59.327000 4044 torch/_logging/_internal.py:1130] [23/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
W0430 23:23:09.801000 4044 torch/_dynamo/convert_frame.py:964] [23/8] torch._dynamo hit config.recompile_limit (8)
W0430 23:23:09.801000 4044 torch/_dynamo/convert_frame.py:964] [23/8]    function: 'wrapper' (/usr/local/lib/python3.11/dist-packages/torch/optim/optimizer.py:465)
W0430 23:23:09.801000 4044 torch/_dynamo/convert_frame.py:964] [23/8]    last reason: 23/7: ___as_tensor(args[0].param_groups[0]['lr']).item() == 0.001926422119140625  # (unknown source ___as_tensor(args[0].param_groups[0]['lr']).item(), please file a bug)
W0430 23:23:09.801000 4044 torch/_dynamo/convert_frame.py:964] [23/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0430 23:23:09.801000 4044 torch/_dynamo/convert_frame.py

(Qwen3ForCausalLM(
   (model): Qwen3Model(
     (embed_tokens): Embedding(151936, 2560)
     (layers): ModuleList(
       (0-35): 36 x Qwen3DecoderLayer(
         (self_attn): Qwen3Attention(
           (q_proj): QuantLinear()
           (k_proj): QuantLinear()
           (v_proj): QuantLinear()
           (o_proj): QuantLinear()
           (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
           (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
         )
         (mlp): Qwen3MLP(
           (gate_proj): QuantLinear()
           (up_proj): QuantLinear()
           (down_proj): QuantLinear()
           (act_fn): SiLU()
         )
         (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
         (post_attention_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
       )
     )
     (norm): Qwen3RMSNorm((2560,), eps=1e-06)
     (rotary_emb): Qwen3RotaryEmbedding()
   )
   (lm_head): Linear(in_features=2560, out_features=151936, bias=False)
 ),
 ['./Qwen3-4B-autoround-2bit-gptq'])

## 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "Qwen/Qwen3-4B"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
from auto_round import AutoRound



autoround = AutoRound(model, tokenizer, nsamples=512, iters=512, low_gpu_mem_usage=False, enable_torch_compile=True, bits=4, seqlen=4096, group_size=128, sym=True)
output_dir = "./Qwen3-4B-autoround-4bit-gptq"
autoround.quantize_and_save(output_dir, format='auto_gptq')



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2025-05-01 00:55:21 INFO autoround.py L240: using torch.bfloat16 for quantization tuning
2025-05-01 00:55:21 INFO autoround.py L570: start to cache block inputs
2025-05-01 00:55:24,000 INFO config.py L54: PyTorch version 2.7.0 available.
2025-05-01 00:55:39 INFO autoround.py L575: caching done
Quantizing model.layers.0:   0%|          | 0/36 [00:03<?, ?it/s]W0501 00:55:56.456000 8540 torch/_logging/_internal.py:1130] [23/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
W0501 00:56:04.159000 8540 torch/_dynamo/convert_frame.py:964] [23/8] torch._dynamo hit config.recompile_limit (8)
W0501 00:56:04.159000 8540 torch/_dynamo/convert_frame.py:964] [23/8]    function: 'wrapper' (/usr/local/lib/python3.11/dist-packages/torch/optim/optimizer.py:465)
W0501 00:56:04.159000 8540 torch/_dynamo/convert_frame.py:964] [23/8]    last reason: 23/7: ___as_tensor(args[0].param_groups[0]['lr']).item() == 0.001926422119140625  # (unknown source ___as_tensor(args[0].pa

(Qwen3ForCausalLM(
   (model): Qwen3Model(
     (embed_tokens): Embedding(151936, 2560)
     (layers): ModuleList(
       (0-35): 36 x Qwen3DecoderLayer(
         (self_attn): Qwen3Attention(
           (q_proj): QuantLinear()
           (k_proj): QuantLinear()
           (v_proj): QuantLinear()
           (o_proj): QuantLinear()
           (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
           (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
         )
         (mlp): Qwen3MLP(
           (gate_proj): QuantLinear()
           (up_proj): QuantLinear()
           (down_proj): QuantLinear()
           (act_fn): SiLU()
         )
         (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
         (post_attention_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
       )
     )
     (norm): Qwen3RMSNorm((2560,), eps=1e-06)
     (rotary_emb): Qwen3RotaryEmbedding()
   )
   (lm_head): Linear(in_features=2560, out_features=151936, bias=False)
 ),
 ['./Qwen3-4B-autoround-4bit-gptq'])